# 🏌️ Mini Caddie — Custom Golf Model Training
Train a YOLOv8n model on the unified golf dataset (ball, club, swing)
Export to ONNX for Hailo compilation


In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# STEP 2: Install YOLOv8
!pip install ultralytics -q

In [ ]:
# STEP 3: Unzip dataset
import os, zipfile
zip_path = '/content/drive/MyDrive/unified-golf-dataset.zip'
extract_path = '/content/dataset'
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)
print('Dataset extracted!')
!ls /content/dataset/unified/
!cat /content/dataset/unified/data.yaml

In [ ]:
# STEP 4: Fix data.yaml paths
yaml_path = '/content/dataset/unified/data.yaml'
with open(yaml_path, 'r') as f:
    c = f.read()
c = c.replace('../train/images', '/content/dataset/unified/train/images')
c = c.replace('../valid/images', '/content/dataset/unified/valid/images')
c = c.replace('../test/images', '/content/dataset/unified/test/images')
with open(yaml_path, 'w') as f:
    f.write(c)
print('Paths updated!')
print(c)

In [ ]:
# STEP 5: Train YOLOv8 (50 epochs)
# For quick test first: change epochs to 5
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
results = model.train(
    data='/content/dataset/unified/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='mini_caddie_golf',
    patience=20,
    save=True,
    save_period=10,
)


In [ ]:
# STEP 6: Export to ONNX for Hailo
best = YOLO('/content/runs/detect/mini_caddie_golf/weights/best.pt')
best.export(format='onnx', opset=11)
print('ONNX exported!')

In [ ]:
# STEP 7: Save models to Google Drive
import shutil
shutil.copy('/content/runs/detect/mini_caddie_golf/weights/best.onnx', '/content/drive/MyDrive/mini_caddie_golf_best.onnx')
shutil.copy('/content/runs/detect/mini_caddie_golf/weights/best.pt', '/content/drive/MyDrive/mini_caddie_golf_best.pt')
print('Models saved to Drive!')

In [ ]:
# STEP 8: Quick test on validation images
results = best.predict(
    source='/content/dataset/unified/valid/images',
    save=True,
    conf=0.25,
    max_det=10,
    project='/content/runs/detect',
    name='mini_caddie_test'
)
print('Test predictions saved!')